##### Import libraries

In [1]:
import tensorflow as tf

##### Load and prepare the MNIST dataset. Convert the samples from integers to floating-point numbers:

In [2]:
mnist = tf.keras.datasets.mnist

(x_train, y_train),(x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0


##### Build the tf.keras.Sequential model by stacking layers. Choose an optimizer and loss function for training:

In [3]:
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(28, 28)),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dropout(0.2),
  tf.keras.layers.Dense(10, activation='softmax')
])

##### For each example the model returns a vector of "logits" or "log-odds" scores, one for each class.

In [4]:
predictions = model(x_train[:1]).numpy()
predictions

array([[0.11242526, 0.12212975, 0.12863109, 0.12220067, 0.05500496,
        0.08544096, 0.14215086, 0.09445403, 0.0265056 , 0.1110568 ]],
      dtype=float32)

##### The tf.nn.softmax function converts these logits to "probabilities" for each class:

In [5]:
tf.nn.softmax(predictions).numpy()

array([[0.10119257, 0.10217938, 0.10284584, 0.10218662, 0.09554574,
        0.09849848, 0.10424574, 0.09939026, 0.09286118, 0.10105419]],
      dtype=float32)

##### Note: It is possible to bake this tf.nn.softmax in as the activation function for the last layer of the network. While this can make the model output more directly interpretable, this approach is discouraged as it's impossible to provide an exact and numerically stable loss calculation for all models when using a softmax output.
##### The `losses.SparseCategoricalCrossentropy` loss takes a vector of logits and a `True` index and returns a scalar loss for each example.


In [6]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

##### This loss is equal to the negative log probability of the true class: It is zero if the model is sure of the correct class.
##### This untrained model gives probabilities close to random (1/10 for each class), so the initial loss should be close to -tf.log(1/10) ~= 2.3.

In [7]:
loss_fn(y_train[:1], predictions).numpy()

2.3177142

In [8]:
model.compile(optimizer='adam',
              loss=loss_fn,
              metrics=['accuracy'])

##### The Model.fit method adjusts the model parameters to minimize the loss:

In [9]:
model.fit(x_train, y_train, epochs=5)

Epoch 1/5
1875/1875 [==============================] - 3s 2ms/step - loss: 1.5863 - accuracy: 0.8921

##### The Model.evaluate method checks the models performance, usually on a "Validation-set" or "Test-set".

In [10]:
model.evaluate(x_test,  y_test, verbose=2)

313/313 - 0s - loss: 1.4928 - accuracy: 0.9693


[1.4928460121154785, 0.9692999720573425]

##### The image classifier is now trained to ~98% accuracy on this dataset.

##### If you want your model to return a probability, you can wrap the trained model, and attach the softmax to it:

In [11]:
probability_model = tf.keras.Sequential([
  model,
  tf.keras.layers.Softmax()
])
probability_model(x_test[:5])

<tf.Tensor: shape=(5, 10), dtype=float32, numpy=
array([[0.08533674, 0.08533674, 0.08533674, 0.08533674, 0.08533674,
        0.08533674, 0.08533674, 0.23196931, 0.08533674, 0.08533674],
       [0.08533674, 0.08533674, 0.23196931, 0.08533674, 0.08533674,
        0.08533674, 0.08533674, 0.08533674, 0.08533674, 0.08533674],
       [0.08533685, 0.2319677 , 0.08533695, 0.08533685, 0.08533686,
        0.08533686, 0.08533689, 0.08533703, 0.08533721, 0.08533685],
       [0.23196934, 0.08533675, 0.08533675, 0.08533675, 0.08533675,
        0.08533675, 0.08533675, 0.08533675, 0.08533675, 0.08533675],
       [0.08533678, 0.08533678, 0.08533678, 0.08533678, 0.2319687 ,
        0.08533678, 0.08533678, 0.08533678, 0.08533678, 0.08533704]],
      dtype=float32)>